# ARTI 308 – Lab 5: Feature Engineering (Classification)
## Password Strength Prediction

### Lab focus
This dataset contains raw passwords and a strength label.  
In this lab, we focus on **feature engineering** for a classification task — extracting meaningful numerical features from raw text so a machine learning model can learn from them.

### Objective
Build a baseline model to predict `strength` (0 = Weak, 1 = Medium, 2 = Strong) and learn how feature engineering choices affect model performance and feature importance.

In this lab we will:
1) Load and inspect the dataset  
2) Engineer new features from raw password text  
3) Encode and prepare features  
4) Train a baseline **Random Forest** classifier  
5) Interpret performance and feature importance  

## 1. Setup and imports

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Load the dataset

In [ ]:
df = pd.read_csv("data.csv", on_bad_lines='skip')
df.head(10)

The dataset contains raw passwords and their strength labels:
- `0` = Weak
- `1` = Medium
- `2` = Strong

Since the only input is a raw text string, we cannot use it directly in a model.  
We must engineer numerical features from the password text.

## 3. Quick dataset checks

In [ ]:
print("Shape:", df.shape)
print("\nMissing values per column:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

## 4. Target variable and class balance

In [ ]:
target_col = "strength"
df[target_col].value_counts()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x=target_col, data=df)
plt.title("Password Strength Distribution")
plt.xlabel("Strength (0=Weak, 1=Medium, 2=Strong)")
plt.ylabel("Count")
plt.show()

This bar chart shows whether the classes are balanced.  
If one class dominates, the model may learn to predict that class more often, so we must interpret accuracy carefully and also look at the confusion matrix.

## 5. Leakage awareness (important)

When designing a prediction task, we must avoid using features that would not be available at prediction time, or features that directly reveal the answer.

In our case, the raw `password` string is our only input — we must derive all features from it.  
We should **not** use the `strength` column as an input feature since it is our target.

## 6. Feature Engineering

### 6.1 Basic length and character-count features
We extract simple numerical properties from each password.

In [ ]:
df_fe = df.copy()
df_fe['password'] = df_fe['password'].astype(str)

# Length of password
df_fe['length'] = df_fe['password'].str.len()

# Number of digits
df_fe['num_digits'] = df_fe['password'].str.count(r'[0-9]')

# Number of uppercase letters
df_fe['num_upper'] = df_fe['password'].str.count(r'[A-Z]')

# Number of lowercase letters
df_fe['num_lower'] = df_fe['password'].str.count(r'[a-z]')

# Number of special characters
df_fe['num_special'] = df_fe['password'].str.count(r'[^a-zA-Z0-9]')

df_fe[['password','length','num_digits','num_upper','num_lower','num_special','strength']].head(10)

We transformed each raw password into meaningful numerical features.  
Models can now learn from these values instead of the raw text string.

### 6.2 Ratio-based features
We create ratio features to capture the proportion of each character type relative to the total password length.  
This allows the model to differentiate passwords of different lengths fairly.

In [ ]:
df_fe['digit_ratio']   = df_fe['num_digits']  / df_fe['length']
df_fe['upper_ratio']   = df_fe['num_upper']   / df_fe['length']
df_fe['special_ratio'] = df_fe['num_special'] / df_fe['length']

df_fe[['password','length','digit_ratio','upper_ratio','special_ratio','strength']].head(10)

`digit_ratio`, `upper_ratio`, and `special_ratio` are derived features that capture the composition of the password.  
These are examples of business-driven feature engineering.

### 6.3 Boolean flag features
We create binary flags indicating whether a password contains at least one of each character type.  
These may capture important patterns for strength classification.

In [ ]:
df_fe['has_digit']   = (df_fe['num_digits']  > 0).astype(int)
df_fe['has_upper']   = (df_fe['num_upper']   > 0).astype(int)
df_fe['has_special'] = (df_fe['num_special'] > 0).astype(int)

df_fe[['password','has_digit','has_upper','has_special','strength']].head(10)

### 6.4 Discretization (binning)

Discretization converts a continuous numerical feature into categories (bins).  
This can help some models capture non-linear relationships, and it also improves interpretability.

Here we discretize `length` into simple tiers.

In [ ]:
df_fe['length_tier'] = pd.cut(
    df_fe['length'],
    bins=[0, 6, 9, 12, np.inf],
    labels=['very_short', 'short', 'medium', 'long']
)

df_fe[['password','length','length_tier','strength']].head(10)

`length_tier` groups numeric values into understandable categories.  
This may help the model capture patterns such as higher weak-password rates for very short passwords.

## 7. Prepare features for modeling

We now select our predictors and drop the raw password string (not directly usable by the model).

In [ ]:
feature_cols = [
    'length', 'num_digits', 'num_upper', 'num_lower', 'num_special',
    'digit_ratio', 'upper_ratio', 'special_ratio',
    'has_digit', 'has_upper', 'has_special'
]

X = df_fe[feature_cols]
y = df_fe[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head()

## 8. Split into train and test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

We use stratified splitting to keep class proportions similar in train and test sets.  
This makes evaluation more reliable for classification problems with imbalanced classes.

## 9. Baseline model (Random Forest)

### Why Random Forest for this lab?
We use Random Forest as a baseline because:
- it handles mixed features well
- it is robust for teaching purposes
- it provides feature importance to help us interpret engineered features

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

rf.fit(X_train, y_train)

## 10. Train the model and evaluate

In [ ]:
y_pred = rf.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=['Weak','Medium','Strong'],
            yticklabels=['Weak','Medium','Strong'])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

Accuracy gives a general sense of performance, but the classification report is more informative.  
Precision answers: when the model predicts a class, how often is it correct?  
Recall answers: out of all real cases of a class, how many did the model find?

The confusion matrix shows which strength classes the model confuses most often.

## 11. Feature Importance (What mattered the most?)

Random Forest provides a built-in feature importance score.  
This helps us understand which engineered features contributed most to predicting password strength.

In [ ]:
fi = (pd.DataFrame({'feature': feature_cols, 'importance': rf.feature_importances_})
        .sort_values('importance', ascending=False))

fi

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=fi, x='importance', y='feature')
plt.title('Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

This chart helps us understand which engineered features contributed most to predicting password strength.  
A high importance score suggests the feature provides useful signal.

## 12. Optional: Feature selection using SelectFromModel

We can select a subset of features using model-based selection.  
This is optional and mainly used to illustrate the concept of feature selection after feature engineering.

In [ ]:
from sklearn.feature_selection import SelectFromModel

selector = SelectFromModel(
    estimator=RandomForestClassifier(
        n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced_subsample"
    ),
    threshold="median"
)

model_fs = Pipeline(steps=[
    ("select", selector),
    ("rf", RandomForestClassifier(
        n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced_subsample"
    ))
])

model_fs.fit(X_train, y_train)
y_pred_fs = model_fs.predict(X_test)

print("Accuracy (with feature selection):", round(accuracy_score(y_test, y_pred_fs), 4))
print("\nClassification Report (with feature selection):")
print(classification_report(y_test, y_pred_fs))

If performance stays similar, feature selection may help simplify the model with minimal accuracy loss.  
If performance drops, it may indicate that important information was removed.

## 13. Student Tasks

### Task 1
Create one new engineered feature that you believe will help predict password strength.  
Write one paragraph justifying your choice.

### Task 2
Try a different binning rule for `length_tier` and discuss whether performance changes.

### Task 3
Remove the ratio features (`digit_ratio`, `upper_ratio`, `special_ratio`) and retrain.  
Compare accuracy and feature importances — were the ratio features helpful?

### Task 4
Run the optional feature selection section and explain whether it was beneficial in your case.

## Wrap-up
In this lab, the dataset contained only raw text, so feature engineering was essential.  
We engineered length-based, count-based, ratio-based, and flag features from password strings, then evaluated a baseline classifier and interpreted feature importance.